# Demo 09d — Latent semantic indexing and collaborative filtering

**Class 09 · E2** (slides: `slides/09_blind_signal_separation.md`)

Two applications of low-rank structure where the "sources" are hidden topics or tastes:
**LSI** factorises a term x document matrix (tf-idf + TruncatedSVD) so that documents
land in a topic space and queries can be answered there; **collaborative filtering**
factorises an incomplete ratings matrix (rwnmf with a NaN mask) to predict the missing
entries.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf

from faa import bss, data, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")

## Slide: LSI by hand — a term x document matrix with two topics

Four documents per topic, nine words; word-document counts. The singular values show
the rank; the left singular vectors are the topics (words), the right the documents.

In [ ]:
words = ["ball", "goal", "kick", "score", "vote", "party", "law", "election", "seat"]
docs = 8
rng = np.random.default_rng(SEED)
X = rng.poisson(0.3, (len(words), docs)).astype(float)  # background counts
X[:4, :4] += rng.poisson(3, (4, 4))  # topic "football" in docs 0-3
X[4:, 4:] += rng.poisson(3, (5, 4))  # topic "politics" in docs 4-7
U, sv, Vt = np.linalg.svd(X - X.mean(0))
slide("LSI toy: term x document matrix")
print(pl.DataFrame(np.c_[words, X], schema=["word", *(f"d{i}" for i in range(docs))]))
print(f"singular values: {sv.round(2)}")

## Slide: LSI toy — rank-2 latent space separates the topics

Project documents onto the top-2 right singular vectors: the two clusters are the two
topics, with no labels used. This is the same picture PCA gives, seen from the document side.

In [ ]:
f, ax = plt.subplots(figsize=(plotlib.COL, 2.2))
P = Vt[:2].T  # document coordinates in the latent space
ax.scatter(P[:4, 0], P[:4, 1], color=BLUE, label="football")
ax.scatter(P[4:, 0], P[4:, 1], color=ORANGE, label="politics")
ax.set_xlabel("latent dim 1"), ax.set_ylabel("latent dim 2")
ax.legend(fontsize=7)
plotlib.save(f, "p4_lsi_toy")
print(f"topic 1 loading of 'ball' vs 'vote': {U[0, 0]:.2f} vs {U[4, 0]:.2f}")
print(f"topic 2 loading of 'ball' vs 'vote': {U[0, 1]:.2f} vs {U[4, 1]:.2f}")

## Slide: LSI on spambase — tf-idf documents in a topic space

Spambase rows are e-mails, columns are word frequencies: a document collection.
tf-idf reweights frequent words down; TruncatedSVD is PCA on the tf-idf matrix
without centering (sparse-friendly). Then a query vector lands in the same space.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfTransformer

Xs, ys, cols = data.load_table("spambase", "spam")
Xtf = TfidfTransformer().fit_transform(Xs)
# row lengths are already unit after tf-idf; keep it simple and fit on the raw rows
svd = TruncatedSVD(n_components=2, random_state=SEED).fit(Xtf)
L = svd.transform(Xtf)
f, ax = plt.subplots(figsize=(plotlib.COL, 2.2))
ax.scatter(L[ys == 0, 0], L[ys == 0, 1], s=4, color=SKY, label="ham")
ax.scatter(L[ys == 1, 0], L[ys == 1, 1], s=4, color=RED, label="spam")
ax.set_xscale("log")  # dim 1 spans two decades: a few very long e-mails stretch it
ax.set_ylim(-0.02, 0.3)  # the bulk of the e-mails lives below dim-2 = 0.3
ax.set_xlabel("latent dim 1 (log scale)"), ax.set_ylabel("latent dim 2")
ax.legend(fontsize=7)
plotlib.save(f, "p4_lsi_spambase")
slide("LSI on spambase (tf-idf + rank 2)")
print(f"explained variance ratio: {svd.explained_variance_ratio_.round(3)} (of {Xs.shape[1]} terms)")

## Slide: Collaborative filtering — the toy ratings matrix

6 users x 8 items, two taste groups in the columns. We hide 25 % of the observed
entries as a test set; the model sees only the rest and is scored on the hidden ones.

In [ ]:
Rfull, Mall = bss.ratings(SEED)  # ratings() NaNs 25 % already: those are never predictable
Mobs = Mall  # observed = trainable
test = np.zeros_like(Mobs)
test[0, [1, 6]] = True  # take a few observed cells out as ground truth
test[3, [1, 4]] = True
test[5, [2, 5]] = True
test[4, [0, 7]] = True
Mtrain = Mobs & ~test
R = Rfull.copy()
R[~Mtrain] = np.nan  # the model sees only the training cells
slide("CF toy: ratings with a train/test split")
print(
    pl.DataFrame(np.where(Mtrain, R, np.nan), schema={f"item{i}": pl.Float64 for i in range(8)}).with_columns(
        user=[f"u{i}" for i in range(6)]
    )
)
print(f"train entries: {Mtrain.sum()}, test entries: {test.sum()}")

## Slide: CF by rwnmf — fill the gaps with a rank-2 factorisation

`rwnmf` ignores the NaN entries through its mask and factorises R ≈ W Hᵀ with
r = 1 (see the sweep below: r = 2 already overfits 29 training cells). The predictions
for the missing cells are the recommendation scores.

In [ ]:
rec, W, H, _ = pynnmf.rwnmf(R, 1, num_iter=500, seed=SEED)
pred = W @ H.T  # rwnmf returns V as (n, k)
slide("rwnmf on the ratings matrix (r = 1)")
print(f"RMSE on the test entries:     {np.sqrt(np.mean((pred[test] - Rfull[test]) ** 2)):.3f}")
print(f"RMSE on the training entries: {np.sqrt(np.mean((pred[Mtrain] - Rfull[Mtrain]) ** 2)):.3f}")
top = np.argsort(-(pred * (~Mtrain)).ravel())[:5]
print("top-5 recommendations (user, item, predicted rating):")
for idx in top:
    u, i = divmod(int(idx), 8)
    print(f"  u{u} -> item {i}: {pred[u, i]:.2f}")

## Slide: Choosing r — the bias-variance curve of the rank

Rank sweep on test RMSE: r = 1 underfits (one taste axis), r ≥ 2 overfits 29
training cells and never recovers. With tiny data, low rank IS the regulariser.

In [ ]:
rows = []
for r_ in [1, 2, 3, 4, 6]:
    rec_r, Wr, Hr, _ = pynnmf.rwnmf(
        R,
        r_,
        num_iter=500,
        seed=SEED,
    )
    pr = Wr @ Hr.T
    rows.append(
        {
            "r": r_,
            "test RMSE": np.sqrt(np.mean((pr[test] - Rfull[test]) ** 2)),
            "train RMSE": np.sqrt(np.mean((pr[Mtrain] - Rfull[Mtrain]) ** 2)),
        }
    )
slide("Rank sweep for collaborative filtering")
print(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(3)))

## Sweep and failure cases (more than the slides)

1. CF with a user who rated nothing: rwnmf still predicts their row from the taste groups
   (only the taste structure, not their own offset — the predictions are small).
2. LSI on spambase: with 57 terms, rank 2 keeps 41 % of the variance but the topics are
   coarse — a real corpus needs thousands of terms for topics to become interpretable.

In [ ]:
slide("Failure / extra: a user with no ratings")
R2, _ = bss.ratings(SEED)
R2[2, :] = np.nan  # user u2 rated nothing
rec2, W2, H2, _ = pynnmf.rwnmf(R2, 1, num_iter=500, seed=SEED)
p2 = W2 @ H2.T
print("user u2 has no ratings; rwnmf still predicts their row from the taste groups")
print(f"predicted u2 row: {p2[2].round(2)}")
print(f"true u2 row:      {Rfull[2].round(2)}")
print(f"test RMSE still computable: {np.sqrt(np.mean((p2[test] - Rfull[test]) ** 2)):.3f}")

slide("Failure / extra: LSI topics are coarse at rank 2")
print(f"spambase rank-2 variance kept: {svd.explained_variance_ratio_.sum():.1%} of {Xs.shape[1]} terms")
print("the two 'topics' separate ham vs spam, not football vs politics: latent axes are")
print("whatever the data varies most, not named concepts — LSI is a projection, not a taxonomy.")